# Team evolution
Create a plan, simulate meetings and inspect the teamwork scores.

In [ ]:
from datetime import datetime
from functools import partial
from getpass import getpass
from pathlib import Path
import json
import os
import sys

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "team_evolution/simulation_config.json").is_file())
sys.path.insert(0, str(root))
from team_evolution.config_loader import load_config
from team_evolution.experiments.engine import load_resources, simulate
from team_evolution.experiments.evolution import prepare_evolution, validate_plan, run_evolution

config = load_config(root / "team_evolution/simulation_config.json")
meeting_model = config["model"]["model_name"]
classifier_model_name = "gpt-6.1-sol"
generation_updates = 10
translate_ja = False
# To resume after restarting the kernel, enter the saved experiment.json path here.
resume_plan = None

## Prepare or reopen a plan
Each new plan draws fresh seeds. Keep the printed path to resume the experiment.

In [ ]:
if resume_plan:
    plan_path = Path(resume_plan).expanduser().resolve()
    plan = json.loads(plan_path.read_text(encoding="utf-8"))
else:
    config["model"]["model_name"] = meeting_model
    plan = prepare_evolution(config, classifier_model_name=classifier_model_name,
                             generation_updates=generation_updates)
    for field in ("translate_actions_ja", "translate_events_ja", "summarize_shared_memories_ja",
                  "summarize_steps_ja", "summarize_result_ja"):
        plan["config"]["replay"][field] = translate_ja
    run_id = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    plan_path = root / "runs" / run_id / "experiment.json"
    validate_plan(plan)
    plan_path.parent.mkdir(parents=True, exist_ok=False)
    plan_path.write_text(json.dumps(plan, indent=2, ensure_ascii=False), encoding="utf-8")
validate_plan(plan)
output = plan_path.parent / "results"
print(plan_path)
print([team["members"] for team in plan["teams"]])

## Run meetings
Start with generation 0. Increase the limit and rerun the last cell to continue.

In [ ]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")
model, classifier_model, embedder = load_resources(plan["config"])
runner = partial(simulate, model=model, classifier_model=classifier_model, embedder=embedder)

In [ ]:
through_generation = 0
records = run_evolution(plan, output, runner, through_generation=through_generation)
print(f"{len(records)} completed meetings")
print(output / "index.html")
print(output / "report.md")